In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 수집, 탐색

In [3]:
df = pd.read_csv('./data/premium.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
df = df.drop_duplicates()
df.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1337 non-null   int64  
 1   sex       1337 non-null   str    
 2   bmi       1332 non-null   float64
 3   children  1337 non-null   int64  
 4   smoker    1337 non-null   str    
 5   region    1337 non-null   str    
 6   charges   1337 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 83.6 KB


# 전처리

중복체크, 널처리, 인코딩

In [5]:
print(df.bmi.isnull())

0       False
1       False
2       False
3       False
4       False
        ...  
1333    False
1334    False
1335    False
1336    False
1337    False
Name: bmi, Length: 1337, dtype: bool


In [6]:
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())

In [7]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [8]:
from sklearn.preprocessing import LabelEncoder

In [9]:
# 문자열 데이터의 수치화 > LabelEncoder
col_list = ['sex', 'smoker', 'region']
for col in col_list: 
  enc = LabelEncoder()
  df[col] = enc.fit_transform(df[col])
  
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


# 분할하기

In [10]:
X = df.iloc[:,:-1]
y = df.iloc[:, -1]

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, random_state=42)

# 스케일링 하기


In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train['bmi'] = scaler.fit_transform(X_train[['bmi']])
X_test['bmi'] = scaler.transform(X_test[['bmi']])
X_test['bmi']


900    -1.330419
1064   -0.818614
1256    0.970629
298     0.639656
237     1.303260
          ...   
534     1.649993
542     0.956527
760     0.671177
1284    0.956527
1285   -1.030968
Name: bmi, Length: 268, dtype: float64

# 선형회귀 모델

In [13]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae, mse, rmse, r2

(4170.6423560015255,
 35550130.517054714,
 np.float64(5962.393019338352),
 0.8065362865570331)

# 다항회귀 모델

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

In [15]:
degree = [2, 3, 4]

for deg in degree:
    model_poly = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('linear', LinearRegression())
    ])
    
    model_poly.fit(X_train, y_train)
    poly_pred = model_poly.predict(X_test)
    
    mae = mean_absolute_error(y_test, poly_pred)
    mse = mean_squared_error(y_test, poly_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, poly_pred)
    
    print(f'Degree {degree} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f} | R^2: {r2:.4f}')

Degree [2, 3, 4] | MAE: 2838.2869 | MSE: 20879449.3187 | RMSE: 4569.4036 | R^2: 0.8864
Degree [2, 3, 4] | MAE: 3021.4520 | MSE: 22521905.9297 | RMSE: 4745.7250 | R^2: 0.8774
Degree [2, 3, 4] | MAE: 3245.7158 | MSE: 26691158.4155 | RMSE: 5166.3487 | R^2: 0.8547


In [16]:
from sklearn.ensemble import RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100, random_state= 0)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_rf)
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_rf)
mae, mse, rmse, r2

(2597.4139290194025,
 21900427.796358738,
 np.float64(4679.7892897393085),
 0.8808179315842309)

In [17]:
import pandas as pd
feature_names = ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
importance_df = pd.DataFrame({
  "feature" : feature_names,
  'importance' : model_rf.feature_importances_
}).sort_values('importance', ascending= False)

print('\n--- 특성 중요도 ---')
print(importance_df)


--- 특성 중요도 ---
    feature  importance
4    smoker    0.600497
2       bmi    0.212468
0       age    0.138651
3  children    0.024268
5    region    0.016831
1       sex    0.007285


In [18]:
#XGBRegressor
from xgboost import XGBRegressor

xgb_reg = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=0
)

xgb_reg.fit(X_train, y_train)
poly_pred = xgb_reg.predict(X_test)

mae = mean_absolute_error(y_test, poly_pred)
mse = mean_squared_error(y_test, poly_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, poly_pred)
    
print(f'MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f} | R^2: {r2:.4f}')

MAE: 2631.0079 | MSE: 19747411.6369 | RMSE: 4443.8060 | R^2: 0.8925


# 하이퍼파라미터 탐색

In [19]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'random_state': 0
    }
    model = RandomForestRegressor(**params)
    score = -cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error').mean()
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print(f'Best MAE : {round(study.best_value, 4)}')
print(f'Best Params : {study.best_params}')

[I 2026-03-03 11:32:32,331] A new study created in memory with name: no-name-f4d93770-3b95-41a7-a3ed-35dfff1d7d8a
[I 2026-03-03 11:32:32,895] Trial 0 finished with value: 2606.317332326284 and parameters: {'n_estimators': 95, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 1}. Best is trial 0 with value: 2606.317332326284.
[I 2026-03-03 11:32:33,646] Trial 1 finished with value: 2569.2395923415215 and parameters: {'n_estimators': 139, 'max_depth': 6, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 1 with value: 2569.2395923415215.
[I 2026-03-03 11:32:33,903] Trial 2 finished with value: 2519.865620792829 and parameters: {'n_estimators': 56, 'max_depth': 4, 'min_samples_split': 4, 'min_samples_leaf': 5}. Best is trial 2 with value: 2519.865620792829.
[I 2026-03-03 11:32:34,975] Trial 3 finished with value: 2605.617143726918 and parameters: {'n_estimators': 165, 'max_depth': 14, 'min_samples_split': 8, 'min_samples_leaf': 2}. Best is trial 2 with value: 2519.865

Best MAE : 2489.8587
Best Params : {'n_estimators': 68, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 5}


In [27]:
from sklearn.ensemble import RandomForestRegressor

best_model = RandomForestRegressor(
    **study.best_params,
    random_state=0
)

best_model.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",68
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",7
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2 : {r2:.4f}")

MAE : 5010.0570
RMSE : 9236.2124
R2 : 0.5358
